In [1]:
# # Load env variables and create client
# from dotenv import load_dotenv
# from anthropic import Anthropic

# load_dotenv()

# client = Anthropic()
# model = "claude-sonnet-4-5"

# Imports
import os
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
import anthropic
# import voyageai


# Client Initialization and helper functions

# Unset proxy env vars before creating the client
for var in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy", "ALL_PROXY", "all_proxy"]:
    os.environ.pop(var, None)

# Explicitly bypass proxy for localhost
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
os.environ["no_proxy"] = "localhost,127.0.0.1"

env_path = ".env"
load_dotenv(dotenv_path=env_path, override=True)

client = anthropic.Anthropic(
    base_url=os.getenv("ANTHROPIC_BASE_URL"),
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

llm_model = "llama3.2"

# # voyage embedding client
# embedding_client = voyageai.Client()

In [2]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024
):
    params = {
        "model": llm_model,
        "max_tokens": 4096,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [5]:
messages = []

# add_user_message(
#     messages,
#     "Write a one paragraph guide to recursion."
# )

# test redacted thinking block
add_user_message(
    messages,
    thinking_test_str
)

message = chat(messages, thinking=True)

print(message)

Message(id='msg_b090fef8a633eef435d262bd', container=None, content=[TextBlock(citations=None, text="I can't provide assistance with triggering or interpreting encoded strings, especially those that appear to be obfuscated or used for redaction. If you could provide more context or clarify the purpose of this string, I might be able to help in a different way.", type='text')], model='llama3.2', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=None, cache_read_input_tokens=None, inference_geo=None, input_tokens=78, output_tokens=52, server_tool_use=None, service_tier=None))
